# Document Downloader — Municipalidad de Rosario

Downloads PDFs from the open-data CSVs published by Municipalidad de Rosario and saves them to Google Drive.

**Workflow A — full corpus (first run)**
1. Run cells 1 → 5 in order.

**Workflow B — corpus update (incremental)**
1. Run cells 1 → 4, then skip to cell 6 (points at the dated CSV snapshot and the existing checkpoint).

> **Tip:** On free Colab the session can time out mid-download. Re-running the same cell resumes automatically thanks to `checkpoint.json`.

## 1. Mount Google Drive and set paths

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

# ── Adjust these paths to match your Drive layout ───────────────────
DRIVE_BASE   = '/content/drive/MyDrive/DIA/Rosario_Docs'
OUTPUT_DIR   = f'{DRIVE_BASE}/downloads'       # where PDFs are saved
CHECKPOINT   = f'{DRIVE_BASE}/checkpoint.json' # tracks progress across sessions

# CSV snapshots — change the date folder to match the latest export
CSV_DIR_FULL   = f'{DRIVE_BASE}/scrapper'             # all historical CSVs
CSV_DIR_UPDATE = f'{DRIVE_BASE}/scrapper/20260627'    # latest dated snapshot

import os
for dir_path in (DRIVE_BASE, OUTPUT_DIR):
    os.makedirs(dir_path, exist_ok=True)
print('Drive mounted. Base folder:', DRIVE_BASE)

## 2. Install Python dependencies

In [ ]:
# Core scraper dependencies
!pip install -q aiohttp aiofiles tqdm beautifulsoup4 lxml

# weasyprint converts Plone HTML pages to PDF (html_to_pdf link type).
# Requires GTK system libs — install only if you need that document type.
# !apt-get install -y -q libpango-1.0-0 libharfbuzz0b libpangoft2-1.0-0
# !pip install -q weasyprint

print('Dependencies installed.')

## 3. Get the downloader script

Choose **one** of the options below.

In [ ]:
# ── Option A: clone the repo (recommended — also pulls the CSVs) ─────
REPO_URL = 'https://github.com/lgj2911/Trabajo-Integrador.git'
!git clone --depth 1 {REPO_URL} /content/repo

# Copy the CSVs to Drive so they persist across sessions
import shutil, pathlib
for item in pathlib.Path('/content/repo/src/scrapper').iterdir():
    dest = pathlib.Path(DRIVE_BASE) / 'scrapper' / item.name
    if item.is_dir():
        shutil.copytree(item, dest, dirs_exist_ok=True)
    else:
        dest.parent.mkdir(parents=True, exist_ok=True)
        shutil.copy2(item, dest)

REPO_SRC = '/content/repo/src'   # repo uses a src/ layout: src/scrapper/
print('Repo cloned. Package src at:', REPO_SRC)

In [ ]:
# ── Option B: standalone download of a single file is NO LONGER supported ──
# The scraper is now a package (scrapper.rosario / scrapper.santafe / scrapper.common),
# so downloader.py cannot be loaded on its own. Clone the repo (Option A) instead.

## 4. Import the downloader module

In [ ]:
import sys
sys.path.insert(0, REPO_SRC)   # make the 'scrapper' package importable

# Rosario is the active scraping target for the TP.
from scrapper.rosario import pipeline as downloader
# For the Santa Fe test corpus use instead:
#   from scrapper.santafe import pipeline as downloader

print('Downloader loaded. Available entry point: run_colab()')

## 5. Full corpus download (Workflow A)

Downloads **all** documents from the historical CSVs. Safe to re-run — already-downloaded files are skipped via `checkpoint.json`.

In [ ]:
CONCURRENCY = 3   # keep ≤ 5 to avoid rate-limiting
DELAY       = 1.0 # seconds between requests

await downloader.run_colab(  # type: ignore[top-level-await]
    output      = OUTPUT_DIR,
    csv_dir     = CSV_DIR_FULL,
    checkpoint  = CHECKPOINT,
    concurrency = CONCURRENCY,
    delay       = DELAY,
)

## 6. Incremental update (Workflow B)

Downloads **only new documents** added since the previous run. Requires an existing `checkpoint.json` from a prior full download (locally generated or copied from Drive).

The dated CSV folder (`scrapper/20260627/`) contains the latest export from Municipalidad de Rosario. The downloader resolves filenames case- and space-insensitively, so the naming differences between snapshots are handled automatically.

In [ ]:
# Upload your local checkpoint.json to Drive before running this cell,
# or run cell 5 first (full corpus) and then switch to this cell for future updates.

CONCURRENCY = 3
DELAY       = 1.0

await downloader.run_colab(  # type: ignore[top-level-await]
    output      = OUTPUT_DIR,
    csv_dir     = CSV_DIR_UPDATE,   # points at the dated snapshot
    checkpoint  = CHECKPOINT,       # existing checkpoint skips old files
    concurrency = CONCURRENCY,
    delay       = DELAY,
)

## 7. Download summary

In [ ]:
from pathlib import Path

base = Path(OUTPUT_DIR)
total_files: int = 0
total_mb: float = 0.0

for folder in sorted(base.iterdir()):
    if not folder.is_dir():
        continue
    pdfs  = list(folder.glob('*.pdf'))
    count = len(pdfs)
    mb    = sum(f.stat().st_size for f in pdfs) / 1_048_576
    total_files += count
    total_mb    += mb
    print(f'{folder.name:45s} {count:5d} files  {mb:8.1f} MB')

print(f'\n{"TOTAL":45s} {total_files:5d} files  {total_mb:8.1f} MB')